### Powerful Multi-Agent AI System with Groq
- This notebook builds a general-purpose multi-agent system that can answer general questions, perform mathematical calculations, and search the web for current information.

#### Install dependencies

In [1]:
!pip install -U langchain-groq langchain langgraph langgraph-supervisor langchain-community ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Foun

#### Import libraries

In [2]:
import os
from getpass import getpass

from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langgraph_supervisor import create_supervisor
from langchain_community.tools import DuckDuckGoSearchRun

/tmp/ipykernel_725/3881274651.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


#### Configure Groq

In [5]:
if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Prajwal Enter Your Key:")

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
)

#### Create web search and mathematical tools

In [6]:
search = DuckDuckGoSearchRun()

def search_web(query: str) -> str:
    """Search the web for current, factual, or topic-specific information."""
    return search.invoke(query)

def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

def subtract(a: float, b: float) -> float:
    """Subtract the second number from the first number."""
    return a - b

def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

def divide(a: float, b: float) -> float:
    """Divide the first number by the second number."""
    if b == 0:
        raise ValueError("Cannot divide by zero.")
    return a / b

def power(a: float, b: float) -> float:
    """Raise the first number to the power of the second number."""
    return a ** b

def modulus(a: float, b: float) -> float:
    """Return the remainder after dividing the first number by the second number."""
    if b == 0:
        raise ValueError("Cannot calculate modulus with zero.")
    return a % b

def percentage(value: float, percent: float) -> float:
    """Calculate a percentage of a value."""
    return (value * percent) / 100

def square_root(value: float) -> float:
    """Calculate the square root of a non-negative number."""
    if value < 0:
        raise ValueError("Cannot calculate the square root of a negative number.")
    return value ** 0.5

#### Create web search and mathematical tools

In [7]:
search = DuckDuckGoSearchRun()

def search_web(query: str) -> str:
    """Search the web for current, factual, or topic-specific information."""
    return search.invoke(query)

def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

def subtract(a: float, b: float) -> float:
    """Subtract the second number from the first number."""
    return a - b

def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

def divide(a: float, b: float) -> float:
    """Divide the first number by the second number."""
    if b == 0:
        raise ValueError("Cannot divide by zero.")
    return a / b

def power(a: float, b: float) -> float:
    """Raise the first number to the power of the second number."""
    return a ** b

def modulus(a: float, b: float) -> float:
    """Return the remainder after dividing the first number by the second number."""
    if b == 0:
        raise ValueError("Cannot calculate modulus with zero.")
    return a % b

def percentage(value: float, percent: float) -> float:
    """Calculate a percentage of a value."""
    return (value * percent) / 100

def square_root(value: float) -> float:
    """Calculate the square root of a non-negative number."""
    if value < 0:
        raise ValueError("Cannot calculate the square root of a negative number.")
    return value ** 0.5


In [12]:
from langchain.agents import create_agent

#### Create the research agent

In [14]:
research_agent = create_agent(
    model=model,
    tools=[search_web],
    system_prompt=(
        "You are a research specialist. "
        "Handle questions that require current, changing, niche, or "
        "web-verifiable information. "
        "Use the web search tool when appropriate and provide a clear answer."
    )
)

research_agent.name = "research_agent"

#### Create the math agent

In [15]:
math_agent = create_agent(
    model=model,
    tools=[
        add,
        subtract,
        multiply,
        divide,
        power,
        modulus,
        percentage,
        square_root
    ],
    system_prompt=(
        "You are a mathematics specialist. "
        "Solve mathematical problems accurately. "
        "Use the available mathematical tools for calculations. "
        "You can use multiple tools for multi-step problems. "
        "Explain the calculation clearly and provide the final answer."
    )
)

math_agent.name = "math_agent"

#### Create the general-purpose agent

In [13]:
from langchain.agents import create_agent

general_agent = create_agent(
    model=model,
    tools=[search_web],
    system_prompt=(
        "You are a general-purpose AI assistant. "
        "Answer general questions about programming, AI, technology, "
        "science, education, business, writing, and everyday topics. "
        "Use the web search tool when current or specific information is required. "
        "Give clear, accurate, and descriptive answers."
    )
)

general_agent.name = "general_agent"

#### Create the supervisor

In [16]:
from langgraph_supervisor import create_supervisor

workflow = create_supervisor(
    [
        general_agent,
        research_agent,
        math_agent
    ],
    model=model,
    prompt=(
        "You are the main supervisor for a general-purpose AI system. "
        "Analyze every user request and delegate it to the appropriate specialist. "
        "Use general_agent for normal questions. "
        "Use research_agent when current, web-based, or niche information is required. "
        "Use math_agent for mathematical and quantitative problems. "
        "For complex questions, use the specialists as needed. "
        "Return one clear and descriptive final answer to the user."
    )
)

#### Compile the application

In [17]:
app = workflow.compile()

#### Test a general question

In [18]:
result = app.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Explain what LangGraph is and why it is useful for AI agents."
        }
    ]
})

print(result["messages"][-1].content)

**LangGraph – What It Is and Why It’s Useful for AI Agents**

---

### 1️⃣ What is LangGraph?

LangGraph is an open‑source Python library (maintained by the LangChain team) that lets you **declare, compose, and run graph‑structured workflows for LLM‑powered agents**.  
Think of it as a lightweight state‑machine / DAG engine built specifically for the patterns that arise when you orchestrate large language models, tools, memory, and external APIs.

Key concepts:

| Concept | Description |
|---------|-------------|
| **Node** | A single callable unit – e.g., an LLM prompt, a tool call, a memory lookup, or any Python function. |
| **Edge** | Directed connection between nodes; can be unconditional, conditional, or loop‑forming. |
| **State** | A mutable dictionary passed along the graph; nodes read/write to it, enabling context sharing. |
| **Control‑flow primitives** | `if`, `while`, `for`, `switch`, `parallel`, `join`, `subgraph`, etc., built‑in to the library. |
| **Visualization** | Gr

#### Test a mathematics question

In [19]:
result = app.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Calculate 20% of 500, multiply the result by 4, and add 50."
        }
    ]
})

print(result["messages"][-1].content)

The calculation proceeds as follows:

1. 20 % of 500 = 0.20 × 500 = **100**  
2. 100 × 4 = **400**  
3. 400 + 50 = **450**

**Result: 450**


#### Test a current-information question

In [20]:
result = app.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What are the latest major developments in generative AI?"
        }
    ]
})

print(result["messages"][-1].content)

**Latest Major Developments in Generative AI (as of September 2026)**  

Below is a concise, up‑to‑date snapshot of the most consequential breakthroughs, new models, applications, and industry trends that are reshaping the field right now. All items are drawn from the latest public announcements, research papers, and reputable news sources (e.g., OpenAI blog, Anthropic press releases, Google I/O, major tech‑news outlets).

---

## 1. New “generation‑6” foundation models  

| Company / Lab | Model (release) | Core capabilities | Notable benchmarks / claims |
|---------------|-----------------|-------------------|------------------------------|
| **OpenAI** | **GPT‑6 “Astra”** – preview 3 Sep 2026, full rollout 4 Sep 2026 | • 1.2 trillion parameters (≈ 30 % larger than GPT‑5.6) <br>• Unified text‑image‑audio‑video generation <br>• Built‑in “agentic” tool‑use (plan‑execute‑replan loop) <br>• Stronger factuality (FACTS‑Score 78.4) <br>• Context window 128 k tokens (≈ 4× GPT‑4‑Turbo) | • 2‑

#### Ask any question

In [21]:
question = input("Prajwal Ask anything: ")

result = app.invoke({
    "messages": [
        {
            "role": "user",
            "content": question
        }
    ]
})

print("\nAnswer:\n")
print(result["messages"][-1].content)

Prajwal Ask anything: Who is Prajwal Ghotkar?

Answer:

Prajwal Ghotkar is a young Indian professional and student who is building a career in data analytics and data science. He is currently a B.E. Computer Science student (expected to graduate in 2025) at Sant Gadge Baba Amravati University in Maharashtra, India, and positions himself as an aspiring data analyst/data‑science enthusiast. His skill set includes Python, SQL, Power BI, Tableau, machine‑learning, data cleaning, exploratory data analysis, predictive modeling, and data visualization. He maintains an active LinkedIn profile and a personal “Data Analysis World” portfolio where he showcases projects and provides a contact email (pmghotkar05@gmail.com). His public tagline is “The world is one big data problem,” and he is actively seeking internships, entry‑level analyst roles, or freelance data‑science projects.
